# 1. Master Table 생성 및 연결 검증

## 1.1 원본 데이터 확인

### 1.1.1 라이브러리와 데이터 불러오기

세 개의 원본 입력 테이블을 불러와 구조와 컬럼 의미를 확인한다. 원본 Excel은 `Data of Ship Block Scheduling in English/Raw_input_ Data/`에서 읽으며, 스케줄링 결과는 같은 데이터셋의 `Result/`에서 불러온다.

In [1]:
#판다스 설치
!pip install pandas


[notice] A new release of pip available: 22.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
#오픈파이엑셀 설치
!pip install openpyxl


[notice] A new release of pip available: 22.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
#엑셀 데이터 읽기
from pathlib import Path
import pandas as pd

DATA_ROOT = Path('Data of Ship Block Scheduling in English')
RAW_ROOT = DATA_ROOT / 'Raw_input_ Data'
RESULT_ROOT = DATA_ROOT / 'Result'
PROCESSED_DIR = Path('data') / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

block = pd.read_excel(RAW_ROOT / 'block_information_table.xlsx') #블록별 생산·일정계획 속성 
position = pd.read_excel(RAW_ROOT / 'block_position_information.xlsx') #작업 위치별 공간·자원 제약 
initial = pd.read_excel(RAW_ROOT / 'initial_block_position_information.xlsx') #블록과 작업 위치의 초기 배정 상태

### 1.1.2 Block table

Block table은 872개 Block과 14개 컬럼으로 구성된다.

- `block position cycle`: 결측 없이 존재하며 이후 회귀분석의 목표값으로 사용한다.
- `block output`: 길이와 폭에서 계산되는 면적 정보와 중복될 가능성이 있다.
- `is combined`, `combined block`: 값이 2건뿐이므로 모델링 단계에서 제외한다.

In [4]:
print(block.shape, '\n')
print(block.info(), '\n')
block.head()

(872, 14) 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 872 entries, 0 to 871
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   index                 872 non-null    int64  
 1   ship no.              872 non-null    object 
 2   block no.             872 non-null    int64  
 3   block type            872 non-null    object 
 4   length                872 non-null    float64
 5   width                 872 non-null    float64
 6   block weight          872 non-null    float64
 7   on-position date      872 non-null    object 
 8   earliest start time   872 non-null    object 
 9   latest start time     872 non-null    object 
 10  block position cycle  872 non-null    int64  
 11  block output          872 non-null    float64
 12  is combined           2 non-null      float64
 13  combined block        2 non-null      float64
dtypes: float64(6), int64(3), object(5)
memory usage: 95.5+ KB
None

,index,ship no.,block no.,block type,length,width,block weight,on-position date,earliest start time,latest start time,block position cycle,block output,is combined,combined block
0,757,H1088,284,flat,20.000,18.400,211.017143,2018-03-03,2018-02-24,2018-06-30,72,368.000000,NaN,NaN
1,758,H1088,294,flat,20.000,18.400,211.017143,2018-03-03,2018-02-24,2018-06-29,71,368.000000,NaN,NaN
2,823,H1088,836,flat,6.450,21.691,146.314852,2018-03-18,2018-03-17,2018-06-04,53,139.906950,NaN,NaN
3,822,H1088,826,flat,20.852,21.691,240.000000,2018-03-18,2018-03-17,2018-06-04,53,452.300732,NaN,NaN
4,819,H1088,101,flat,30.660,9.400,240.000000,2018-03-25,2018-03-18,2018-06-11,61,288.204000,NaN,NaN


### 1.1.3 Position table

Position table은 66개 작업 위치와 13개 컬럼으로 구성된다. 위치의 구역, 크기, 인양 능력, 높이 제한, 작업팀, 허용 Block 유형 등을 담고 있어 Block과 결합했을 때 작업 위치의 물리적·운영적 조건을 설명할 수 있다.

In [5]:
print(position.shape, '\n')
print(position.info(), '\n')
position.head()

(66, 13) 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66 entries, 0 to 65
Data columns (total 13 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   index                       66 non-null     int64  
 1   primary area                66 non-null     object 
 2   secondary area              66 non-null     object 
 3   block position id           66 non-null     object 
 4   block position description  66 non-null     object 
 5   size                        66 non-null     object 
 6   lifting capacity (T)        66 non-null     int64  
 7   height limit (M)            11 non-null     object 
 8   labor team                  66 non-null     object 
 9   designated block            6 non-null      float64
 10  block type                  36 non-null     object 
 11  attribute                   66 non-null     object 
 12  column1                     2 non-null      object 
dtypes: float64(1), int64(2), o

,index,primary area,secondary area,block position id,block position description,size,lifting capacity (T),height limit (M),labor team,designated block,block type,attribute,column1
0,1,curved surface,48m,BQM4801A,48 South-1,5*10,240,9,MZ,110.0,NaN,fixed,NaN
1,2,curved surface,48m,BQM4802A,48 South-2,5*10,240,9,MZ,110.0,NaN,fixed,NaN
2,3,curved surface,48m,BQM4803A,48 South-3,36*21,240,9,MZ,NaN,NaN,general,NaN
3,8,curved surface,48m,BQM4808A,48 South-8,20*14,240,9,MZ,111.0,NaN,fixed,NaN
4,10,curved surface,48m,BQM4810A,48 South-10,22*21,240,9,MZ,NaN,NaN,general,NaN


### 1.1.4 Initial table

Initial table은 35개의 초기 상태를 담고 있다. 예측 대상 Block의 본 테이블이 아니라 스케줄링 시작 시점의 점유 상태와 초기 제약을 정의하는 별도 입력이다. 뒤에서 연결 가능성을 검증하되, 현재 회귀모델에는 직접 포함하지 않는다.

In [6]:
print(initial.shape, '\n')
print(initial.info(), '\n')
initial.head()

(35, 6) 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35 entries, 0 to 34
Data columns (total 6 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   ship no.                    35 non-null     object        
 1   vessel type                 35 non-null     float64       
 2   block no.                   35 non-null     int64         
 3   earliest start time         35 non-null     datetime64[ns]
 4   block position description  35 non-null     object        
 5   expected completion time    35 non-null     datetime64[ns]
dtypes: datetime64[ns](2), float64(1), int64(1), object(2)
memory usage: 1.8+ KB
None 



,ship no.,vessel type,block no.,earliest start time,block position description,expected completion time
0,H1156,17.7,825,2017-10-23,48 North-1,2017-11-10
1,H1156,17.7,835,2017-10-23,48 North-2,2017-11-10
2,H1094,17.7,825,2017-10-25,48 North-3,2017-11-15
3,H1094,17.7,835,2017-10-25,48 North-4,2017-11-15
4,H1095,17.7,825,2017-10-25,48 North-5,2017-11-15


### 1.1.5 기본 확인 결과와 데이터 흐름

| 테이블 | 행 수 | 핵심 확인 |
|---|---:|---|
| Block | 872 | 기준 작업기간 8~72일, 평균 약 26.93일 |
| Position | 66 | 물리·자원·조직 특성 포함, 일부 구조적 결측 존재 |
| Initial | 35 | 스케줄링 시작 시점의 초기 상태를 기록 |

Block과 Position table에는 직접적인 배정 정보가 없다. 따라서 실제로 어떤 Block이 어떤 Position에 배치되었는지를 담은 **스케줄링 Result를 중간 연결표**로 사용한다.

여기서 이후 해석에 중요한 구조적 사실을 하나 확인할 수 있다. `block position cycle`은 알고리즘별 Result가 아니라 **스케줄링 입력인 Block table에 이미 존재한다.** 반면 선택 Position, 계획 시작·종료시점과 Delay는 Result에 기록된다. 이 단계에서는 Cycle의 정확한 산정 방식까지 단정하지 않고, 입력값과 결과값의 위치가 다르다는 사실만 확인한다.

## 1.2 스케줄링 Result와 연결 키 검증

### 1.2.1 일곱 가지 스케줄링 결과

Result 폴더에는 두 개의 강화학습 기반 결과와 다섯 개의 휴리스틱 결과가 있다.

| 구분 | 방법 |
|---|---|
| 강화학습 | DDQN, EDDQN |
| 휴리스틱 | Earliest Start Time, Longest Processing Time, Resource Utilization Based, Response Time Based, Shortest Processing Time |

먼저 일곱 결과의 컬럼 구조가 동일한지 확인하고, 현재 분석에서는 DDQN을 대표 결과로 사용한다.

In [7]:
result_DDQN = pd.read_excel(RESULT_ROOT / 'DDQN scheduling.xlsx')
result_EDDQN = pd.read_excel(RESULT_ROOT / 'EDDQN_scheduling_results.xlsx')
result_earliest= pd.read_excel(RESULT_ROOT / 'scheduling_results_heuristic_earliest_start_time_20250605_014601.xlsx')
result_longest = pd.read_excel(RESULT_ROOT / 'scheduling_results_heuristic_longest_processing_time_20250605_014601.xlsx')
result_utilization = pd.read_excel(RESULT_ROOT / 'scheduling_results_heuristic_resource_utilization_based_20250605_014601.xlsx')
result_response = pd.read_excel(RESULT_ROOT / 'scheduling_results_heuristic_response_time_based_20250605_014601.xlsx')
result_shortest = pd.read_excel(RESULT_ROOT / 'scheduling_results_heuristic_shortest_processing_time_20250605_014601.xlsx')

In [8]:
#일곱 테이블의 컬럼 구조가 동일
print(result_DDQN.columns.equals(result_EDDQN.columns))
print(result_DDQN.columns.equals(result_earliest.columns))
print(result_DDQN.columns.equals(result_longest.columns))
print(result_DDQN.columns.equals(result_utilization.columns))
print(result_DDQN.columns.equals(result_response.columns))
print(result_DDQN.columns.equals(result_shortest.columns))

True
True
True
True
True
True


In [9]:
print(result_DDQN.shape, '\n')
print(result_DDQN.info(), '\n')
result_DDQN.head()

(872, 18) 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 872 entries, 0 to 871
Data columns (total 18 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   block position description  872 non-null    object        
 1   block position id           872 non-null    object        
 2   block position length       872 non-null    int64         
 3   block position width        872 non-null    int64         
 4   labor team                  872 non-null    object        
 5   ship no.                    872 non-null    object        
 6   block sequence no.          872 non-null    int64         
 7   block no.                   872 non-null    int64         
 8   block length                872 non-null    float64       
 9   block width                 872 non-null    float64       
 10  block processing time       872 non-null    int64         
 11  block output value          872 non-null    fl

,block position description,block position id,block position length,block position width,labor team,ship no.,block sequence no.,block no.,block length,block width,block processing time,block output value,planned start time,planned finish time,latest completion time,delay days,advance days,normal advance days
0,No.10 South-5,PPT1003A,20,20,yard ZH,H1087,1,262,14.600,17.280,39,252.288000,2019-05-27,2019-07-05,2019-06-05,30,-84,6
1,No.10 North-16,PPT1058A,20,20,yard HN,H1087,2,272,14.400,17.280,39,248.832000,2019-04-09,2019-05-18,2019-06-05,-18,-36,6
2,No.10 South-7,PPT1004A,20,20,yard ZH,H1087,4,154,19.932,15.976,47,318.433632,2019-05-06,2019-06-22,2019-06-06,16,-77,8
3,No.8 North-16,PPT0858A,20,22,HL,H1087,5,261,20.100,16.000,37,321.600000,2019-06-21,2019-07-28,2019-05-25,64,-124,6
4,No.8 North-16,PPT0858A,20,22,HL,H1087,6,271,17.800,16.000,37,284.800000,2019-04-06,2019-05-13,2019-05-25,-12,-48,6


### 1.2.2 Block과 Result 연결

`block no.`는 97개 고유값만 있어 하나의 Block을 유일하게 식별하지 못한다. 반면 Block의 `index`와 Result의 `block sequence no.`는 각각 872개 고유값을 가지며 값의 집합도 같다.

조인 후 양쪽 `block no.`가 100% 일치하는지까지 확인하여 연결의 타당성을 검증한다.

In [10]:
# block['block no.']와 result_DDQN['block no.'] 테이블의 'block no.' 유니크한 항목 개수 및 항목 일치여부 확인
print(block['block no.'].nunique())
print(result_DDQN['block no.'].nunique())
set(block['block no.']) == set(result_DDQN['block no.'])

97
97


True

In [11]:
# block['index']와 result_DDQN['block sequence no.']의 유니크한 항목 개수 및 항목 일치여부 확인
print(block['index'].nunique())
print(result_DDQN['block sequence no.'].nunique()) 
set(result_DDQN['block sequence no.']) == set(block['index'])

872
872


True

In [12]:
#index와 block sequence no.로 조인했을 때 정말 같은 블록끼리 붙었는지 검증. false가 섞여있으면 mean값이 1이 아닐테니 완전일치함. 
check = block.merge(
    result_DDQN,
    left_on='index',
    right_on='block sequence no.',
    how='inner',
    suffixes=('_block', '_result')
)

(check['block no._block'] == check['block no._result']).mean()


np.float64(1.0)

#### 1.2.2.1 검증 결론

- 연결 키: `block.index` ↔ `Result.block sequence no.`
- 양쪽 키 모두 872개 고유값
- 키 값의 집합이 서로 일치
- 조인 후 `block no.` 일치율 100%

따라서 Block과 Result는 위 키를 이용해 **1:1로 연결**할 수 있다.

### 1.2.3 Result와 Position 연결

Result의 모든 `block position id`가 Position table에 존재하는지 확인하고, 조인 후 Position 설명이 일치하는지 교차 검증한다.

In [13]:
result_DDQN['block position id'].isin(position['block position id'])

0      True
1      True
2      True
3      True
4      True
       ... 
867    True
868    True
869    True
870    True
871    True
Name: block position id, Length: 872, dtype: bool

In [14]:
check_position = result_DDQN.merge(
    position,
    on='block position id',
    how='inner',
    suffixes=('_result', '_position')
)

(check_position['block position description_result']
 ==
 check_position['block position description_position']).mean()

np.float64(1.0)

#### 1.2.3.1 검증 결론

- Result의 모든 Position ID가 Position table에 존재한다.
- 조인 후 Position 설명 일치율은 100%다.
- 하나의 Position에 여러 Block이 배정될 수 있으므로 **Result 기준으로는 N:1, Position 기준으로는 1:N** 관계다.

### 1.2.4 Initial table 연결 가능성 검토

Initial table을 본 분석에 활용할 수 있는지 두 가지 연결 후보를 검토한다.

1. `block position description`을 이용한 Position–Initial 연결
2. `ship no.` + `block no.` 복합키를 이용한 Block–Initial 연결

README에는 명확한 조인키가 제시되지 않았다. 따라서 단순히 같은 이름의 컬럼을 사용하는 것이 아니라, 각 테이블의 값이 현실에서 동일한 Block 또는 Position을 의미하는지를 기준으로 판단한다.

#### 1.2.4.1 Position–Initial 연결 후보

양쪽 테이블에 존재하는 `block position description`을 이용해, Position이 Initial table에 기록되어 있는지 확인한다.

In [15]:
print(initial.shape, '\n')
print(initial.info(), '\n')
initial.head()

(35, 6) 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35 entries, 0 to 34
Data columns (total 6 columns):
 #   Column                      Non-Null Count  Dtype         
---  ------                      --------------  -----         
 0   ship no.                    35 non-null     object        
 1   vessel type                 35 non-null     float64       
 2   block no.                   35 non-null     int64         
 3   earliest start time         35 non-null     datetime64[ns]
 4   block position description  35 non-null     object        
 5   expected completion time    35 non-null     datetime64[ns]
dtypes: datetime64[ns](2), float64(1), int64(1), object(2)
memory usage: 1.8+ KB
None 



,ship no.,vessel type,block no.,earliest start time,block position description,expected completion time
0,H1156,17.7,825,2017-10-23,48 North-1,2017-11-10
1,H1156,17.7,835,2017-10-23,48 North-2,2017-11-10
2,H1094,17.7,825,2017-10-25,48 North-3,2017-11-15
3,H1094,17.7,835,2017-10-25,48 North-4,2017-11-15
4,H1095,17.7,825,2017-10-25,48 North-5,2017-11-15


In [16]:
# 각 Position이 초기 시점에 이미 점유되어 있었는지 여부 확인
# Position Table의 'block position description'이 Initial Table에 존재하면 True(초기 점유), 존재하지 않으면 False(초기 비점유)
print(position['block position description'].isin(initial['block position description']).value_counts())
# 총 11개의 포지션이 이미 점유되어 있었음.

block position description
False    55
True     11
Name: count, dtype: int64


> **검토 결론**

Initial table은 스케줄링 시작 시점에 이미 작업이 진행 중이던 Position 상태를 기록한 데이터로 해석한다.

- Initial에 존재하는 `block position description`: 초기 점유 상태로 간주
- Initial에 존재하지 않는 `block position description`: 초기 비점유 상태로 간주

다만 공개 데이터만으로는 Initial에 없는 모든 Position이 실제로 비어 있었다고 확정할 수 없다. 따라서 이는 본 분석을 위한 가정이며, 실제 야드 전체의 점유율로 일반화하지 않는다.

#### 1.2.4.2 Block–Initial 연결 후보

`ship no.` + `block no.` 조합이 각 테이블의 Block을 식별할 수 있는지 확인하고, 두 테이블 사이의 실제 매칭률을 검토한다.

In [17]:
# Initial Table에서 'ship no.' + 'block no.' 조합의 중복을 제거한 뒤 고유한 조합의 개수 확인
# → 두 컬럼의 조합이 Initial의 각 Block을 고유하게 식별할 수 있는 복합키 후보인지 검토
print(initial[['ship no.', 'block no.']].drop_duplicates().shape[0])


# Block Table에서도 동일하게 'ship no.' + 'block no.' 조합의 고유 개수 확인
# → Initial과 동일한 복합키를 이용해 Block Table과 직접 연결할 수 있는지 검토
print(block[['ship no.', 'block no.']].drop_duplicates().shape[0])

# Initial과 Block Table에 'ship no.' + 'block no.'를 결합한 동일한 형식의 복합키 생성
# → 두 테이블에서 동일한 Block이 실제로 매칭되는지 확인하기 위한 연결 키로 사용
initial['ship_block_key'] = (initial['ship no.'].astype(str) + '_' + initial['block no.'].astype(str))
block['ship_block_key'] = (block['ship no.'].astype(str) + '_' + block['block no.'].astype(str))

35
174


In [18]:
# Initial Table의 각 복합키(ship_block_key)가 Block Table에도 존재하는지 확인하고,
# 그 평균을 계산하여 Initial Block 중 Block Table과 매칭되는 비율 확인
initial['ship_block_key'].isin(block['ship_block_key']).mean()

np.float64(0.0)

> **검토 결론**

`ship no.` + `block no.` 조합은 각 테이블 내부에서는 복합키 후보가 될 수 있지만, Initial과 Block table 사이의 실제 매칭률은 **0%**다.

- Initial ↔ Block 직접 연결: 불가
- Initial 정보 활용: `block position description`을 이용한 Position 수준의 간접 연결

이는 Initial table에 기록된 Block이 현재 스케줄링 대상 Block이 아니라, 시작 시점에 이미 작업 중이던 별도 Block이라는 해석과 일치한다.

## 1.3 DDQN Result 결합표 생성

앞서 검증한 연결 관계를 이용해 Block, DDQN Result, Position, Initial 정보를 하나의 **탐색용 결합표**로 구성한다.

1. Block `index` ↔ DDQN `block sequence no.`
2. DDQN `block position id` ↔ Position `block position id`
3. Position `block position description` ↔ Initial `block position description`

> **결합표 구조**  
> Block 입력 특성 + DDQN이 선택한 Position 결과 + 해당 Position의 초기 상태

이 표는 조인 검증, EDA와 사후 연관성 분석에는 유용하지만, DDQN이 선택한 Position을 스케줄링 이전의 Block 고유 특성으로 바꾸지는 않는다. 따라서 최종 Cycle 예측모델은 이 표에서 Block 고유 특성만 사용한다.


### 1.3.1 Master Table 연결 ERD

아래 ERD는 Master Table 생성에 실제로 사용한 세 연결키와 현재 데이터에서 확인된 관계를 단순화해 보여준다.

![Master Table 연결 ERD](Docs/images/1_master_table_join_erd.svg)

- **① Block ↔ DDQN Result**: `index`와 `block sequence no.`를 기준으로 연결한다. 현재 데이터에서는 두 키가 모두 고유하여 **1:1** 관계이다.
- **② DDQN Result ↔ Position**: `block position id`를 기준으로 연결한다. 여러 Block 배정 결과가 하나의 Position을 참조하는 **N:1** 관계이다.
- **③ Position ↔ Initial**: `block position description`이 일치할 때만 연결되는 **선택적 0..1** 관계이다. 현재 Position 66개 중 11개가 Initial과 일치하며, 일치 여부를 `initially_occupied`로 표현한다.


In [19]:
# ============================================================
# 데이터 경로 설정 및 원본 테이블 불러오기
# ============================================================

from pathlib import Path
import pandas as pd


# 프로젝트 원본 데이터 폴더 경로 설정
DATA_ROOT = Path('Data of Ship Block Scheduling in English')
RAW_ROOT = DATA_ROOT / 'Raw_input_ Data'
RESULT_ROOT = DATA_ROOT / 'Result'

# 전처리 결과를 저장할 폴더 생성
PROCESSED_DIR = Path('data') / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# Raw Input Table 불러오기
# ============================================================

# Block별 생산·일정계획 속성
block = pd.read_excel(RAW_ROOT / 'block_information_table.xlsx')

# 작업 Position별 공간·자원·조직 관련 제약 및 특성
position = pd.read_excel(RAW_ROOT / 'block_position_information.xlsx')

# 스케줄링 시작 시점의 Block-Position 초기 배정 상태
initial = pd.read_excel(RAW_ROOT / 'initial_block_position_information.xlsx')

In [20]:
#먼저 DDQN 결과에서 `block sequence no.`와 `block position id`만 추출
mapping = result_DDQN[['block sequence no.', 'block position id']].copy()

In [21]:
#추출한 내용 확인
print(mapping.info(), '\n')
mapping.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 872 entries, 0 to 871
Data columns (total 2 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   block sequence no.  872 non-null    int64 
 1   block position id   872 non-null    object
dtypes: int64(1), object(1)
memory usage: 13.8+ KB
None 



,block sequence no.,block position id
0,1,PPT1003A
1,2,PPT1058A
2,4,PPT1004A
3,5,PPT0858A
4,6,PPT0858A


In [22]:
# 1. Result에서는 Block ↔ Position 연결에 필요한 키만 추출
mapping = result_DDQN[
    ['block sequence no.', 'block position id']
].copy()


# 2. Block + DDQN 연결
master = block.merge(
    mapping,
    left_on='index',
    right_on='block sequence no.',
    how='left',
    validate='1:1'
)


# 3. DDQN이 선택한 Position 정보 연결
master = master.merge(
    position,
    on='block position id',
    how='left',
    suffixes=('_block', '_position'),
    validate='m:1'
)


# 4. Initial Table을 Position description 기준으로 연결
master = master.merge(
    initial,
    on='block position description',
    how='left',
    suffixes=('', '_initial')
)


# 5. 최종 Master Table 확인
print(master.shape, '\n')
print(master.info(), '\n')
master.head()

(872, 33) 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 872 entries, 0 to 871
Data columns (total 33 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   index_block                  872 non-null    int64         
 1   ship no.                     872 non-null    object        
 2   block no.                    872 non-null    int64         
 3   block type_block             872 non-null    object        
 4   length                       872 non-null    float64       
 5   width                        872 non-null    float64       
 6   block weight                 872 non-null    float64       
 7   on-position date             872 non-null    object        
 8   earliest start time          872 non-null    object        
 9   latest start time            872 non-null    object        
 10  block position cycle         872 non-null    int64         
 11  block output                 872 

,index_block,ship no.,block no.,block type_block,length,width,block weight,on-position date,earliest start time,latest start time,...,labor team,designated block,block type_position,attribute,column1,ship no._initial,vessel type,block no._initial,earliest start time_initial,expected completion time
0,757,H1088,284,flat,20.000,18.400,211.017143,2018-03-03,2018-02-24,2018-06-30,...,HL,NaN,flat,general,NaN,NaN,NaN,NaN,NaT,NaT
1,758,H1088,294,flat,20.000,18.400,211.017143,2018-03-03,2018-02-24,2018-06-29,...,RH,NaN,NaN,general,NaN,NaN,NaN,NaN,NaT,NaT
2,823,H1088,836,flat,6.450,21.691,146.314852,2018-03-18,2018-03-17,2018-06-04,...,HL,NaN,NaN,general,NaN,NaN,NaN,NaN,NaT,NaT
3,822,H1088,826,flat,20.852,21.691,240.000000,2018-03-18,2018-03-17,2018-06-04,...,MZ,NaN,NaN,general,NaN,H1098,17.7,153.0,2017-10-20,2017-11-08
4,819,H1088,101,flat,30.660,9.400,240.000000,2018-03-25,2018-03-18,2018-06-11,...,ZH,NaN,flat,general,NaN,H1103,31.6,183.0,2017-10-26,2017-11-16


### 1.3.2 분석 의미와 해석상의 주의점

이 결합표는 다음 탐색 질문에 사용한다.

> DDQN이 선택한 Position과 초기 상태를 사후에 붙였을 때, 사전 Block Position Cycle에 대한 통계적 설명력은 어떻게 변하는가?

데이터 생성 순서는 `Block / Position 후보 / Initial / Cycle → DDQN → 선택 Position·계획일정·Delay`다.

여기서 Position Information Table 자체는 스케줄링 입력조건이지만, 특정 Block과 연결된 선택 Position은 DDQN의 결과다. 따라서 선택 Position은 Cycle 예측 시점에 독립적으로 주어진 Block 특성이 아니다.

> **기술적으로 Join이 가능하다는 것과 분석적으로 독립변수로 사용할 수 있다는 것은 별개의 문제다.**

Position·Initial 변수의 중요도와 Set B/C 성능은 사후 연관성 진단으로만 읽는다. 이를 Position의 인과효과나 Scheduling Method의 성과로 해석하지 않는다.


## 1.4 변수 정리와 Master Table 저장

### 1.4.1 분석용 컬럼 정리

Block, Position, Initial의 출처가 드러나도록 컬럼명을 정리한다. 추적용 식별자와 모델링 변수를 구분하고, 종속변수 이름은 `block_processing_cycle`로 통일한다.

In [23]:
# ============================================================
# Master Table 분석용 정리
# - Block / Position / Initial 출처가 명확하도록 컬럼명 변경
# - 분석 및 검토가 쉽도록 컬럼 순서 재정렬
# - 모든 컬럼을 우선 유지
# ============================================================

master = (
    master
    .rename(columns={

        # ===== Block 식별 / 기본 정보 =====
        'index_block': 'block_index',
        'ship no.': 'block_ship_no',
        'block no.': 'block_no',
        'block type_block': 'block_type',

        # ===== Block 물리적 특성 =====
        'length': 'block_length',
        'width': 'block_width',
        'block weight': 'block_weight',

        # ===== Block 일정 정보 =====
        'on-position date': 'block_on_position_date',
        'earliest start time': 'block_earliest_start',
        'latest start time': 'block_latest_start',

        # ===== Block 기타 정보 =====
        'is combined': 'block_is_combined',
        'combined block': 'block_combined_block',

        # ===== Result 연결 정보 =====
        'block sequence no.': 'result_block_sequence_no',

        # ===== Position 식별 / 기본 정보 =====
        'block position id': 'position_id',
        'index_position': 'position_index',
        'block position description': 'position_description',

        # ===== Position 위치 정보 =====
        'primary area': 'position_primary_area',
        'secondary area': 'position_secondary_area',

        # ===== Position 물리 / 자원 정보 =====
        'size': 'position_size',
        'lifting capacity (T)': 'position_lifting_capacity',
        'height limit (M)': 'position_height_limit',
        'labor team': 'position_labor_team',

        # ===== Position 제약 / 속성 =====
        'designated block': 'position_designated_block',
        'block type_position': 'position_block_type',
        'attribute': 'position_attribute',
        'column1': 'position_column1',

        # ===== Initial 식별 / 기본 정보 =====
        'ship no._initial': 'initial_ship_no',
        'vessel type': 'initial_vessel_type',
        'block no._initial': 'initial_block_no',

        # ===== Initial 일정 정보 =====
        'earliest start time_initial': 'initial_earliest_start',
        'expected completion time': 'initial_expected_completion',

        # ===== 종속변수 =====
        'block position cycle': 'block_processing_cycle'
    })
    [[
        # --------------------------------
        # 1. 식별 / 추적용
        # --------------------------------
        'block_index',
        'block_ship_no',
        'block_no',
        'result_block_sequence_no',
        'position_id',
        'position_index',
        'position_description',

        # --------------------------------
        # 2. Block 정보
        # --------------------------------
        'block_type',
        'block_length',
        'block_width',
        'block output',
        'block_weight',
        'block_is_combined',
        'block_combined_block',

        # --------------------------------
        # 3. Block 일정 정보
        # --------------------------------
        'block_on_position_date',
        'block_earliest_start',
        'block_latest_start',

        # --------------------------------
        # 4. Position 정보
        # --------------------------------
        'position_primary_area',
        'position_secondary_area',
        'position_size',
        'position_lifting_capacity',
        'position_height_limit',
        'position_labor_team',
        'position_designated_block',
        'position_block_type',
        'position_attribute',
        'position_column1',

        # --------------------------------
        # 5. Initial 정보
        # --------------------------------
        'initial_ship_no',
        'initial_vessel_type',
        'initial_block_no',
        'initial_earliest_start',
        'initial_expected_completion',

        # --------------------------------
        # 6. Target
        # --------------------------------
        'block_processing_cycle'
    ]]
    .copy()
)


# 결과 확인
master.info()
master.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 872 entries, 0 to 871
Data columns (total 33 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   block_index                  872 non-null    int64         
 1   block_ship_no                872 non-null    object        
 2   block_no                     872 non-null    int64         
 3   result_block_sequence_no     872 non-null    int64         
 4   position_id                  872 non-null    object        
 5   position_index               872 non-null    int64         
 6   position_description         872 non-null    object        
 7   block_type                   872 non-null    object        
 8   block_length                 872 non-null    float64       
 9   block_width                  872 non-null    float64       
 10  block output                 872 non-null    float64       
 11  block_weight                 872 non-null    

,block_index,block_ship_no,block_no,result_block_sequence_no,position_id,position_index,position_description,block_type,block_length,block_width,...,position_designated_block,position_block_type,position_attribute,position_column1,initial_ship_no,initial_vessel_type,initial_block_no,initial_earliest_start,initial_expected_completion,block_processing_cycle
0,757,H1088,284,757,PPT0811A,94,No.8 South-21,flat,20.000,18.400,...,NaN,flat,general,NaN,NaN,NaN,NaN,NaT,NaT,72
1,758,H1088,294,758,PPT1057A,78,No.10 North-14,flat,20.000,18.400,...,NaN,NaN,general,NaN,NaN,NaN,NaN,NaT,NaT,71
2,823,H1088,836,823,PPT0809A,92,No.8 South-17,flat,6.450,21.691,...,NaN,NaN,general,NaN,NaN,NaN,NaN,NaT,NaT,53
3,822,H1088,826,822,BQM4810A,10,48 South-10,flat,20.852,21.691,...,NaN,NaN,general,NaN,H1098,17.7,153.0,2017-10-20,2017-11-08,53
4,819,H1088,101,819,BQM4503A,22,45 South-3,flat,30.660,9.400,...,NaN,flat,general,NaN,H1103,31.6,183.0,2017-10-26,2017-11-16,61


### 1.4.2 Master Table 저장

정리한 Master Table을 `master_DDQN.csv`로 저장하여 이후 전처리 및 EDA 단계에서 불러와 사용한다.

In [24]:
# ============================================================
# Master Table 저장
# - 이후 전처리 및 EDA 단계에서 불러와 사용
# ============================================================

master.to_csv(
    PROCESSED_DIR / 'master_DDQN.csv',
    index=False,
    encoding='utf-8-sig'
)

print(f"{PROCESSED_DIR / 'master_DDQN.csv'} 저장 완료")

data\processed\master_DDQN.csv 저장 완료


### 1.4.3 결합표 소결론

- Block과 Result는 검증된 1:1 키로 연결했다.
- Result와 Position은 Position ID로 연결했다.
- Initial은 현재 Block과 직접 연결되지 않아 Position의 초기 기록 여부로만 활용한다.
- `Block Position Cycle`은 Result보다 먼저 주어진 Block Table의 입력값이다.
- DDQN Position이 포함된 `master_DDQN.csv`는 탐색·결과 비교용 결합표이며 최종 예측 입력표가 아니다.
- 최종 Cycle 예측에는 스케줄링 이전에 알 수 있는 Block 고유 특성만 사용한다.

다음 2번 노트북에서는 Block 변수와 DDQN 결과층 변수를 구분한 채 전처리·EDA를 수행한다.
